# KDCL-MinLogit vs. other distillation methods on NetzTest (4G/5G)

**Place this notebook in the project ROOT** (same level as `src/`, `data/`, `experiments/`), not inside `notebooks/`, so the relative paths below work as-is.

**Data:** needs `netztest-4g5g-2024-2025-sample200k.csv` (25MB). Locally, drop it in `data/raw/` and nothing else is needed. On Colab, upload it to your Drive and set the path in the first code cell below -- it auto-detects Colab and mounts Drive for you.

Run cells top to bottom with **Run All**. Everything (logs, table, chart) stays saved inside this `.ipynb` file after you save it -- that's what makes the results visible/shareable without re-running anything.

Recommended first pass: set `EPOCHS = 5` below just to confirm everything runs end-to-end on your machine, then bump to `EPOCHS = 50` for the real numbers.


In [ ]:
import sys, os, subprocess

# --- Data location: works locally (file already in data/raw/) or on Colab (Drive-mounted) ---
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Edit this to wherever you uploaded netztest-4g5g-2024-2025-sample200k.csv on your Drive
    os.environ['KDCL_RAT_CSV'] = '/content/drive/MyDrive/kdcl-rat-selection/netztest-4g5g-2024-2025-sample200k.csv'

print('Working directory:', os.getcwd())
assert os.path.isdir('src'), (
    'Run this notebook from the project ROOT (the folder containing src/, data/, experiments/). '
    'On Colab: !git clone the repo first, then %cd into it before running this cell.'
)
sys.path.insert(0, os.path.abspath('src'))

import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('(CPU is completely fine for these small MLPs -- no GPU required.)')

csv_path = os.environ.get('KDCL_RAT_CSV', 'data/raw/netztest-4g5g-2024-2025-sample200k.csv')
assert os.path.isfile(csv_path), f'Data file not found at {csv_path} -- fix the path above.'
print('Using data file:', csv_path)


## 1. Configuration
Edit these, then run the training cell below.

In [ ]:
SEEDS = [1, 2, 3]      # repeated-seed runs -- Objective 1 needs this, not a single seed
EPOCHS = 5             # <-- start small (e.g. 5) to smoke-test, then set to 50 for real numbers
RUN_OFFLINE_KD = True  # requires a vanilla run to exist first; handled automatically below

## 2. Train: vanilla, dml, kdcl (minlogit), kdcl (linear), one -- each seed
This calls `src/train.py` as a subprocess per (method, seed) pair, same as `src/run_seeds.py`, but streams output into the notebook so you can watch progress live.


In [ ]:
METHODS = [
    ('vanilla',     ['--mode', 'vanilla']),
    ('dml',         ['--mode', 'dml']),
    ('kdcl',        ['--mode', 'kdcl', '--ensemble', 'minlogit', '--T', '1.0', '--alpha', '0.1']),
    ('kdcl_linear', ['--mode', 'kdcl', '--ensemble', 'linear', '--T', '1.0', '--alpha', '0.1']),
    ('one',         ['--mode', 'one']),
]

def run(cmd):
    print('>', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-2000:])  # tail of output, keeps the notebook readable
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError(f'Command failed: {" ".join(cmd)}')

total = len(METHODS) * len(SEEDS)
done = 0
for label, extra_args in METHODS:
    for seed in SEEDS:
        done += 1
        print(f'\n=== [{done}/{total}] {label}, seed={seed} ===')
        run([sys.executable, 'src/train.py', *extra_args,
             '--epochs', str(EPOCHS), '--seed', str(seed), '--print_freq', str(EPOCHS)])

## 3. Train: offline_kd (needs a vanilla checkpoint as the frozen teacher)

In [ ]:
import glob

if RUN_OFFLINE_KD:
    vanilla_runs = sorted(glob.glob('experiments/netztest_vanilla/*'))
    if not vanilla_runs:
        print('No vanilla run found yet -- run the training cell above first.')
    else:
        teacher_ckpt = vanilla_runs[-1]  # most recent vanilla run
        print('Using teacher checkpoint:', teacher_ckpt)
        for seed in SEEDS:
            print(f'\n=== offline_kd, seed={seed} ===')
            run([sys.executable, 'src/train.py', '--mode', 'offline_kd',
                 '--teacher_ckpt', teacher_ckpt,
                 '--epochs', str(EPOCHS), '--seed', str(seed), '--print_freq', str(EPOCHS)])

## 4. Evaluate on the held-out test set (auto-averages across seeds)

In [ ]:
import evaluate
import importlib
importlib.reload(evaluate)

rows = evaluate.main()


## 5. Results table

In [ ]:
import pandas as pd

results_df = pd.DataFrame(rows)[['mode', 'branch', 'n_runs', 'val_acc_mean', 'val_acc_std', 'test_acc_mean', 'test_acc_std']]
results_df = results_df.sort_values(['branch', 'test_acc_mean'], ascending=[True, False]).reset_index(drop=True)
results_df

In [ ]:
os.makedirs('results', exist_ok=True)
results_df.to_csv('results/summary.csv', index=False)
print('Saved to results/summary.csv')

## 6. Chart: test accuracy by method, per branch

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

branches = results_df['branch'].unique()
fig, axes = plt.subplots(1, len(branches), figsize=(6 * len(branches), 5), sharey=True)
if len(branches) == 1:
    axes = [axes]

for ax, branch in zip(axes, branches):
    sub = results_df[results_df['branch'] == branch].sort_values('test_acc_mean', ascending=False)
    ax.bar(sub['mode'], sub['test_acc_mean'], yerr=sub['test_acc_std'], capsize=5)
    ax.set_title(branch)
    ax.set_ylabel('Test accuracy (%)')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('results/test_accuracy_comparison.png', dpi=150)
plt.show()